# AI Resume Analyzer & Mock Interviewer
Upload a resume (`PDF`, `DOCX`, `TXT`, or `Image`), enter a target job role, review the ATS analysis report, and complete a 10-question mock interview using voice or text answers.

### Cell 1: Install Required Libraries

In [ ]:
!pip install -q google-genai pypdf python-docx pillow gTTS SpeechRecognition gradio

### Cell 2: Import Libraries and Set Up Gemini

In [ ]:
import os
import json
import gradio as gr
import speech_recognition as sr
from PIL import Image
from pypdf import PdfReader
from docx import Document
from gtts import gTTS
from google import genai
from google.genai import types
from google.colab import files, userdata

api_key = userdata.get("Gemini_API_Key_2")
client = genai.Client(api_key=api_key)
model_name = "gemini-3.5-flash"

### Cell 3: Voice Conversion Functions

In [ ]:
def make_audio(msg, out_file):
    plain = msg.replace("*", "").replace("#", "").strip()
    gTTS(text=plain, lang="en", tld="co.in").save(out_file)
    return out_file


def read_voice(wav_file):
    if not wav_file:
        return ""
    rec = sr.Recognizer()
    try:
        with sr.AudioFile(wav_file) as src:
            return rec.recognize_google(rec.record(src))
    except sr.UnknownValueError:
        return ""

### Cell 4: Read Resume File Function

In [ ]:
def get_resume(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".txt":
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read().strip()
    elif ext == ".pdf":
        pdf = PdfReader(path)
        return "\n".join(p.extract_text() or "" for p in pdf.pages).strip()
    elif ext == ".docx":
        doc = Document(path)
        lines = [p.text for p in doc.paragraphs if p.text.strip()]
        cells = [
            c.text.strip()
            for t in doc.tables
            for r in t.rows
            for c in r.cells
            if c.text.strip()
        ]
        return "\n".join(lines + cells).strip()
    elif ext in [".png", ".jpg", ".jpeg", ".webp"]:
        with Image.open(path) as pic:
            return pic.copy()
    raise ValueError("Unsupported file format")

### Cell 5: Upload Resume and Enter Job Role

In [ ]:
print("Upload your resume (PDF, DOCX, TXT, or Image):")
up = files.upload()

if up:
    fname = list(up.keys())[0]
    resume_data = get_resume(fname)
    print("Successfully uploaded:", fname)

    role = input("Enter target job role: ").strip()
    while not role:
        role = input("Please enter a valid target job role: ").strip()
else:
    print("No file uploaded.")

### Cell 6: Resume Analysis (ATS Score, Missing Skills, Improvement Topics)

In [ ]:
if "resume_data" in locals() and "role" in locals():
    prompt = f"""
    You are a technical recruiter and ATS resume reviewer.
    Check the attached resume for the role: {role}.

    Give a clear report with these 3 sections:
    1. ATS Score: Score out of 100 with a short reason.
    2. Missing Skill Set: Important skills, tools, or frameworks for {role} that are missing or weak.
    3. Improvement in Topics: Specific technical topics, project updates, and resume sections to improve.
    """

    print("Analyzing resume for:", role)
    res = client.models.generate_content(
        model=model_name,
        contents=[prompt, resume_data],
        config=types.GenerateContentConfig(temperature=0.3),
    )

    print("RESUME ANALYSIS REPORT -", role)
    print(res.text)
else:
    print("Please run Cell 5 first to upload your resume and enter the target job role.")

### Cell 7: Start 10-Question Interview and Get Question 1

In [ ]:
if "resume_data" in locals() and "role" in locals():
    max_q = 10
    chat = client.chats.create(
        model=model_name,
        config=types.GenerateContentConfig(
            temperature=0.4,
            system_instruction=(
                f"You are a technical interviewer taking an interview for {role}. "
                "Ask short, clear questions based only on the skills and projects in the resume. "
                "Keep questions plain for audio playback without markdown symbols or bullet points."
            ),
        ),
    )

    q1 = chat.send_message(
        [
            resume_data,
            f"Ask Question 1 of {max_q} for {role} from the resume skills. Give only the question.",
        ]
    ).text.strip()
    q1_mp3 = make_audio(f"Question 1. {q1}", "q_1.mp3")

    session = {"step": 1, "q": q1, "log": []}
    print("Interview session ready.")
else:
    print("Please run Cell 5 first.")

### Cell 8: Check Answer and Load Next Question

In [ ]:
def next_step(mic_file, box_text):
    if session["step"] > max_q:
        return "Interview completed.", None, "See your final scorecard below.", None, ""

    ans = box_text.strip() if box_text and box_text.strip() else read_voice(mic_file)
    if not ans:
        return (
            f"Q{session['step']}/{max_q}: {session['q']}",
            None,
            "No answer detected. Please record your voice or type your answer.",
            None,
            "",
        )

    step = session["step"]
    last = step == max_q

    check_prompt = f"""
    Question {step}: "{session['q']}"
    Candidate Answer: "{ans}"

    Grade out of 10 for {role}. Return only valid JSON:
    {{
      "marks": <integer 0 to 10>,
      "analysis": "<1 to 2 short spoken sentences>",
      "next_question": "{'DONE' if last else f'Ask Question {step + 1} of {max_q} from the resume skills (plain text only).'}"
    }}
    """

    reply = chat.send_message(
        check_prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0.3),
    )
    out = json.loads(reply.text)
    mark = out.get("marks", 0)
    note = out.get("analysis", "")

    session["log"].append(
        {
            "step": step,
            "q": session["q"],
            "ans": ans,
            "mark": mark,
            "note": note,
        }
    )

    if last:
        session["step"] += 1
        scored = sum(item["mark"] for item in session["log"])
        total = max_q * 10
        pct = round((scored / total) * 100, 2)

        report = f"FINAL SCORECARD: {scored}/{total} ({pct}%)\n\n"
        for item in session["log"]:
            report += f"Q{item['step']}: {item['q']}\nAnswer: {item['ans']}\nScore: {item['mark']}/10 | {item['note']}\n\n"

        end_mp3 = make_audio(
            f"You scored {mark} out of 10. Interview complete. Your total score is {scored} out of {total}.",
            "final.mp3",
        )
        return "Interview Completed", end_mp3, report, None, ""

    nxt_q = out.get("next_question", "").strip()
    session["step"] += 1
    session["q"] = nxt_q

    nxt_mp3 = make_audio(
        f"You scored {mark} out of 10. {note}. Question {session['step']}. {nxt_q}",
        f"q_{session['step']}.mp3",
    )
    info = f"Your Answer: {ans}\nScore: {mark}/10 - {note}"
    return f"Q{session['step']}/{max_q}: {nxt_q}", nxt_mp3, info, None, ""

### Cell 9: Launch Interview UI

In [ ]:
with gr.Blocks() as app:
    gr.Markdown(f"## AI Mock Interviewer ({role})")
    q_view = gr.Textbox(value=f"Q1/{max_q}: {q1}", label="Question", interactive=False)
    q_play = gr.Audio(value=q1_mp3, label="Interviewer Audio", autoplay=True)

    with gr.Row():
        mic = gr.Audio(sources=["microphone"], type="filepath", label="Voice Answer")
        txt = gr.Textbox(label="Text Answer", placeholder="Type your answer here...", lines=3)

    btn = gr.Button("Submit Answer", variant="primary")
    out_box = gr.Textbox(label="Feedback & Final Scorecard", lines=12, interactive=False)

    btn.click(
        fn=next_step,
        inputs=[mic, txt],
        outputs=[q_view, q_play, out_box, mic, txt],
    )

app.launch(inline=True)